# ListenBrainz BigQuery Staging Data Quality Checks

This notebook runs data quality checks on the dbt staging table:

`my-project-sssint1.listenbrainz_gcp.stg_listenbrainz_listen`

The checks validate required fields, uniqueness, blank text values, and future timestamps.


## 1. Install required packages

Run this cell first if the packages are not installed.


In [1]:
!pip install google-cloud-bigquery pandas db-dtypes


## 2. Configure BigQuery connection

Update the service account key path to match your own computer.


In [2]:
import os

# Change this path to your actual service account JSON key file
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"/home/sansa/my-project-sssint1-7e02e9078e06.json"

PROJECT_ID = "my-project-sssint1"
DATASET = "listenbrainz_gcp"
TABLE = "stg_listenbrainz_listen"


FULL_TABLE = f"`{PROJECT_ID}.{DATASET}.{TABLE}`"
print("Validating table:", FULL_TABLE)

TABLE = "fact_listening_events"
FULL_TABLE = f"`{PROJECT_ID}.{DATASET}.{TABLE}`"
print("Validating table:", FULL_TABLE)

Validating table: `my-project-sssint1.listenbrainz_gcp.stg_listenbrainz_listen`
Validating table: `my-project-sssint1.listenbrainz_gcp.fact_listening_events`


## 3. Test BigQuery connection

This checks that Python can connect to BigQuery and read the staging table.


In [3]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)

query = f"""
SELECT COUNT(*) AS total_rows
FROM {FULL_TABLE}
"""

row_count = client.query(query).to_dataframe()
row_count


/home/sansa/miniconda3/envs/elt/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,total_rows
0,1902187797


## 4. Define data quality checks

These checks are Great Expectations-style validations using BigQuery SQL.


In [10]:
checks = {
    "Required fields are not null": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM {FULL_TABLE}
        WHERE listen_id IS NULL
           OR listened_at IS NULL
           OR user_name IS NULL
           OR artist_name IS NULL
           OR track_name IS NULL
           OR recording_msid IS NULL
    """,

    "listen_id is unique": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM (
            SELECT listen_id
            FROM {FULL_TABLE}
            GROUP BY listen_id
            HAVING COUNT(*) > 1
        )
    """,

    "track_name is not blank": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM {FULL_TABLE}
        WHERE TRIM(track_name) = ''
    """,

    "artist_name is not blank": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM {FULL_TABLE}
        WHERE TRIM(artist_name) = ''
    """,

    "listened_at is not in the future": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM {FULL_TABLE}
        WHERE listened_at > CURRENT_TIMESTAMP()
    """
}


## 5. Run data quality checks


In [11]:
import pandas as pd

results = []

for check_name, sql in checks.items():
    df = client.query(sql).to_dataframe()
    invalid_rows = int(df.loc[0, "invalid_rows"])
    status = "PASS" if invalid_rows == 0 else "FAIL"

    results.append({
        "check_name": check_name,
        "invalid_rows": invalid_rows,
        "status": status
    })

dq_results = pd.DataFrame(results)
dq_results


/home/sansa/miniconda3/envs/elt/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,check_name,invalid_rows,status
0,Required fields are not null,0,PASS
1,listen_id is unique,373057,FAIL
2,track_name is not blank,7,FAIL
3,artist_name is not blank,48,FAIL
4,listened_at is not in the future,0,PASS


## 6. Show failed checks only


In [12]:
failed_checks = dq_results[dq_results["status"] == "FAIL"]
failed_checks


,check_name,invalid_rows,status
1,listen_id is unique,373057,FAIL
2,track_name is not blank,7,FAIL
3,artist_name is not blank,48,FAIL


## 7. Optional: View invalid rows for required fields

Run this only if the required-field check fails.


In [13]:
invalid_required_fields = f"""
SELECT *
FROM {FULL_TABLE}
WHERE listen_id IS NULL
   OR listened_at IS NULL
   OR user_name IS NULL
   OR artist_name IS NULL
   OR track_name IS NULL
   OR recording_msid IS NULL
LIMIT 20
"""

client.query(invalid_required_fields).to_dataframe()


/home/sansa/miniconda3/envs/elt/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,listen_id,listened_at,user_name,artist_msid,artist_name,artist_mbids,release_msid,release_name,release_mbid,recording_msid,recording_mbid,track_name,tags,listen_date,listen_hour,listen_minute


## 8. Report summary

Great Expectations-style data quality checks were performed on the dbt staging table `stg_listenbrainz_listen` in BigQuery. The checks validated that key fields such as `listen_id`, `listened_at`, `user_name`, `artist_name`, `track_name`, and `recording_msid` were not null. Additional checks confirmed that `listen_id` was unique, track and artist names were not blank, and listening timestamps were not in the future. These validations help ensure that the staging data is reliable before it is transformed into dimension and fact tables for analytics.
